# Data Cleaning and ETL Workflow for IT Asset Dataset

This notebook processes an IT asset dataset including asset status, assignment, warranty, maintenance, and compliance information.

The goal is to clean and standardize the data to ensure consistency and make it ready for analysis.

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the raw dataset
# Path is relative to the notebooks folder
try:
    df = pd.read_csv('../data/raw_it_assets.csv')
    print("✅ Data loaded successfully.")
except FileNotFoundError:
    print("❌ File not found. Please check the data folder.")

✅ Data loaded successfully.


## Raw Data Overview

In [3]:
# Initial data inspection
df.head()

,Asset_ID,Asset_Type,Assigned_To,Department,Acquisition_Date,Warranty_Expiry_Date,Status,Maintenance_Cost,License_Expiry_Date,Vendor_Name,Location,Compliance_Status,OS_Version,Repair_Count
0,ASSET_0001,Laptop,User_066,Finance,2025-10-29,2028-10-29,In Use,2994.97,2028-02-29,Dell,Other,Compliant,Windows 11,0
1,ASSET_0002,Monitor,User_228,IT,2020-12-20,2024-06-20,In Use,401.83,NaN,Lenovo,Hsinchu,Non-Compliant,NaN,3
2,ASSET_0003,Server,NaN,R&D,2023-09-04,2028-09-04,Stock,247.42,2027-07-04,Dell,Other,At Risk,VMware ESXi,3
3,ASSET_0004,Tablet,User_327,Sales,2024-02-25,2026-08-25,In Use,164.24,2025-05-25,Apple,Other,Compliant,iPadOS 18,2
4,ASSET_0005,Server,User_263,R&D,2023-04-04,2027-04-04,In Use,1749.18,2026-08-04,Dell,Cloud,Compliant,Ubuntu 22.04,2


In [4]:
df.shape

(1000, 14)

In [5]:
# Check data types and overall structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Asset_ID              1000 non-null   object 
 1   Asset_Type            1000 non-null   object 
 2   Assigned_To           783 non-null    object 
 3   Department            1000 non-null   object 
 4   Acquisition_Date      1000 non-null   object 
 5   Warranty_Expiry_Date  1000 non-null   object 
 6   Status                1000 non-null   object 
 7   Maintenance_Cost      1000 non-null   float64
 8   License_Expiry_Date   780 non-null    object 
 9   Vendor_Name           1000 non-null   object 
 10  Location              1000 non-null   object 
 11  Compliance_Status     1000 non-null   object 
 12  OS_Version            843 non-null    object 
 13  Repair_Count          1000 non-null   int64  
dtypes: float64(1), int64(1), object(12)
memory usage: 109.5+ KB


## Data Cleaning

### 3.1 Data Type Conversion

Convert key columns to appropriate data types for cleaning and analysis.

### 3.2 Missing Value Overview

Before handling missing data, it is important to understand:

- The proportion of missing values  
- Which variables are affected  
- Patterns of missingness  

This helps determine the most appropriate imputation strategy.

In [6]:
# Strip whitespace from string columns just in case
df['Status'] = df['Status'].str.strip()
df['Department'] = df['Department'].str.strip()

In [7]:
# Date Conversion
date_cols = ['Acquisition_Date', 'Warranty_Expiry_Date', 'License_Expiry_Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col].astype(str).str.strip(), errors='coerce', format='mixed')

# 2. Categorical Optimization
cat_cols = ['Asset_Type', 'Department', 'Status', 'Location', 'Vendor_Name', 'Compliance_Status']
for col in cat_cols:
    df[col] = df[col].astype('category')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Asset_ID              1000 non-null   object        
 1   Asset_Type            1000 non-null   category      
 2   Assigned_To           783 non-null    object        
 3   Department            1000 non-null   category      
 4   Acquisition_Date      1000 non-null   datetime64[ns]
 5   Warranty_Expiry_Date  1000 non-null   datetime64[ns]
 6   Status                1000 non-null   category      
 7   Maintenance_Cost      1000 non-null   float64       
 8   License_Expiry_Date   780 non-null    datetime64[ns]
 9   Vendor_Name           1000 non-null   category      
 10  Location              1000 non-null   category      
 11  Compliance_Status     1000 non-null   category      
 12  OS_Version            843 non-null    object        
 13  Repair_Count       

In [8]:
# Check for missing values explicitly
df.isnull().sum()

Asset_ID                  0
Asset_Type                0
Assigned_To             217
Department                0
Acquisition_Date          0
Warranty_Expiry_Date      0
Status                    0
Maintenance_Cost          0
License_Expiry_Date     220
Vendor_Name               0
Location                  0
Compliance_Status         0
OS_Version              157
Repair_Count              0
dtype: int64

### 3.3 Missing Value Handling

Missing values are handled using rule-based logic:

- Unassigned assets remain empty (no artificial labels added)
- Missing maintenance cost is treated as zero
- Monitor devices do not have OS versions
- Missing warranty dates are estimated from acquisition date

### 3.4 Logical Consistency Checks

Check for invalid values and inconsistent relationships across columns.

Missing maintenance cost is interpreted as no recorded maintenance expense.

In [9]:
# Fill basic categorical and numerical nulls

# Fill Maintenance_Cost: Empty values suggest no repair costs incurred yet.
df['Maintenance_Cost'] = df.groupby('Asset_Type', observed=False)['Maintenance_Cost'].transform(lambda x: x.fillna(x.mean()))

# Fill OS_Version: Non-computing devices (like monitors) don't have an OS.
no_os_assets = ['Monitor']
df.loc[df['Asset_Type'].isin(no_os_assets), 'OS_Version'] = \
    df.loc[df['Asset_Type'].isin(no_os_assets), 'OS_Version'].fillna('Not Applicable')

# License is not required for non-computing devices such as monitors.
df.loc[df['Asset_Type'].isin(no_os_assets), 'License_Expiry_Date'] = pd.NaT

Warranty is estimated as 3 years from acquisition date when missing.

In [10]:
# Logic-based imputation for Warranty
# Estimate expiry as 3 years from the Acquisition Date

mask = df['Warranty_Expiry_Date'].isnull()
df.loc[mask, 'Warranty_Expiry_Date'] = df.loc[mask, 'Acquisition_Date'] + pd.Timedelta(days=1095)

In [11]:
# Diagnosis: Outliers & Logical Errors
# Repair Count Outliers

print("Repair count > 15:", (df['Repair_Count'] > 15).sum())
df[df['Repair_Count'] > 10][['Asset_ID', 'Repair_Count']].head()

Repair count > 15: 0


,Asset_ID,Repair_Count
84,ASSET_0085,12
113,ASSET_0114,12
304,ASSET_0305,11
602,ASSET_0603,12


In [12]:
# Retired Assets Still Assigned
retired_conflict = df[(df['Status'].isin(['Retired', 'Decommissioned'])) & 
                      (df['Assigned_To'].notna())]
                      
# Clear assignee for retired/decommissioned assets
mask = df['Status'].isin(['Retired', 'Decommissioned'])
df.loc[mask, 'Assigned_To'] = np.nan

print(f"Conflict count: {len(retired_conflict)}")
retired_conflict[['Asset_ID', 'Status', 'Assigned_To']].head()

Conflict count: 20


,Asset_ID,Status,Assigned_To
5,ASSET_0006,Retired,User_353
11,ASSET_0012,Retired,User_087
99,ASSET_0100,Retired,User_226
223,ASSET_0224,Retired,User_071
240,ASSET_0241,Retired,User_261


In [13]:
# OS Version on Non-Computing Assets
monitor_os = df[(df['Asset_Type'] == 'Monitor') & (df['OS_Version'] != 'Not Applicable')]
print(f"Monitors with unexpected OS values: {len(monitor_os)}")

Monitors with unexpected OS values: 0


In [14]:
# Data Rectification

# Fix Repair_Count: Replace 99 with Median
repair_median = df.loc[df['Repair_Count'] != 99, 'Repair_Count'].median()
df.loc[df['Repair_Count'] == 99, 'Repair_Count'] = repair_median

In this dataset, a value of 99 in Repair_Count is treated as an invalid placeholder.

In [15]:
# Check for duplicate Asset IDs
duplicates = df.duplicated(subset=['Asset_ID']).sum()
print(f"Duplicate Asset IDs found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates(subset=['Asset_ID'])

Duplicate Asset IDs found: 0


## 4. Feature Engineering

The following features are created:


- Asset_Age
- Repair_Intensity
- Days_to_Warranty_Expiry
- Is_High_Cost

In [16]:
# --- Feature Engineering for Predictive Analytics ---

# 1. Define Current Reference Date
# Use 'today' as the baseline for all age and expiry calculations
current_date = pd.to_datetime('today')

# 2. Normalize Asset Age to Years
# Convert the duration from acquisition to years for better model interpretability
df['Asset_Age'] = ((current_date - pd.to_datetime(df['Acquisition_Date'])).dt.days / 365).round(1)

# 3. Calculate Repair Intensity (Stress Score)
# Ratio of repairs per year of service; high intensity indicates unreliable assets
# Use .replace(0, 0.1) to prevent division by zero for brand new assets
df['Repair_Intensity'] = (df['Repair_Count'] / df['Asset_Age'].replace(0, 0.1)).round(2)

# 4. Warranty & Operational Risk Lead Indicators
# Calculate days until warranty expiry; negative values represent out-of-warranty risks
df['Days_to_Warranty_Expiry'] = (pd.to_datetime(df['Warranty_Expiry_Date']) - current_date).dt.days
df['Warranty_Status'] = df['Days_to_Warranty_Expiry'].apply(lambda x: 'Expired' if x < 0 else 'Active')

# 5. Define ML Target: High-Cost Flag (Top 25%)
# We treat 'Maintenance_Cost' as the cumulative expenditure per unique Asset ID.
# Assets exceeding the 75th percentile are flagged as high-financial-risk (Target = 1).
cost_threshold = df['Maintenance_Cost'].quantile(0.75)
df['Is_High_Cost'] = (df['Maintenance_Cost'] > cost_threshold).astype(int)

df[['Acquisition_Date', 'Warranty_Expiry_Date', 'Asset_Age', 'Days_to_Warranty_Expiry', 'Warranty_Status', 'Is_High_Cost', 'Repair_Intensity']].head()

,Acquisition_Date,Warranty_Expiry_Date,Asset_Age,Days_to_Warranty_Expiry,Warranty_Status,Is_High_Cost,Repair_Intensity
0,2025-10-29,2028-10-29,0.5,922,Active,1,0.00
1,2020-12-20,2024-06-20,5.3,-670,Expired,0,0.57
2,2023-09-04,2028-09-04,2.6,867,Active,0,1.15
3,2024-02-25,2026-08-25,2.2,126,Active,0,0.91
4,2023-04-04,2027-04-04,3.0,348,Active,1,0.67


In [17]:
df['Asset_Age'].describe()

count    1000.000000
mean        2.662100
std         1.586553
min         0.000000
25%         1.300000
50%         2.500000
75%         3.900000
max         6.300000
Name: Asset_Age, dtype: float64

In [18]:
print("Negative maintenance cost:", (df['Maintenance_Cost'] < 0).sum())
print("Negative asset age:", (df['Asset_Age'] < 0).sum())
print("Warranty earlier than acquisition:",
      (df['Warranty_Expiry_Date'] < df['Acquisition_Date']).sum())

Negative maintenance cost: 0
Negative asset age: 0
Warranty earlier than acquisition: 0


In [19]:
# Organize Columns for Better Readability
column_order = [
    'Asset_ID', 'Asset_Type', 'Status', 'Department', 'Assigned_To',
    'Acquisition_Date', 'Asset_Age',
    'Warranty_Expiry_Date', 'Days_to_Warranty_Expiry', 'Warranty_Status',
    'License_Expiry_Date',
    'Maintenance_Cost', 'Repair_Count', 
    'Repair_Intensity', 
    'Is_High_Cost', 
    'Vendor_Name', 'Location', 'Compliance_Status', 'OS_Version'
]

# Apply the new column order
df = df[column_order]

df.head()

,Asset_ID,Asset_Type,Status,Department,Assigned_To,Acquisition_Date,Asset_Age,Warranty_Expiry_Date,Days_to_Warranty_Expiry,Warranty_Status,License_Expiry_Date,Maintenance_Cost,Repair_Count,Repair_Intensity,Is_High_Cost,Vendor_Name,Location,Compliance_Status,OS_Version
0,ASSET_0001,Laptop,In Use,Finance,User_066,2025-10-29,0.5,2028-10-29,922,Active,2028-02-29,2994.97,0,0.00,1,Dell,Other,Compliant,Windows 11
1,ASSET_0002,Monitor,In Use,IT,User_228,2020-12-20,5.3,2024-06-20,-670,Expired,NaT,401.83,3,0.57,0,Lenovo,Hsinchu,Non-Compliant,Not Applicable
2,ASSET_0003,Server,Stock,R&D,NaN,2023-09-04,2.6,2028-09-04,867,Active,2027-07-04,247.42,3,1.15,0,Dell,Other,At Risk,VMware ESXi
3,ASSET_0004,Tablet,In Use,Sales,User_327,2024-02-25,2.2,2026-08-25,126,Active,2025-05-25,164.24,2,0.91,0,Apple,Other,Compliant,iPadOS 18
4,ASSET_0005,Server,In Use,R&D,User_263,2023-04-04,3.0,2027-04-04,348,Active,2026-08-04,1749.18,2,0.67,1,Dell,Cloud,Compliant,Ubuntu 22.04


## 5. Final Data Overview

In [20]:
df.shape

(1000, 19)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Asset_ID                 1000 non-null   object        
 1   Asset_Type               1000 non-null   category      
 2   Status                   1000 non-null   category      
 3   Department               1000 non-null   category      
 4   Assigned_To              763 non-null    object        
 5   Acquisition_Date         1000 non-null   datetime64[ns]
 6   Asset_Age                1000 non-null   float64       
 7   Warranty_Expiry_Date     1000 non-null   datetime64[ns]
 8   Days_to_Warranty_Expiry  1000 non-null   int64         
 9   Warranty_Status          1000 non-null   object        
 10  License_Expiry_Date      780 non-null    datetime64[ns]
 11  Maintenance_Cost         1000 non-null   float64       
 12  Repair_Count             1000 non-n

In [22]:
df.describe(include='all')

,Asset_ID,Asset_Type,Status,Department,Assigned_To,Acquisition_Date,Asset_Age,Warranty_Expiry_Date,Days_to_Warranty_Expiry,Warranty_Status,License_Expiry_Date,Maintenance_Cost,Repair_Count,Repair_Intensity,Is_High_Cost,Vendor_Name,Location,Compliance_Status,OS_Version
count,1000,1000,1000,1000,763,1000,1000.000000,1000,1000.000000,1000,780,1000.00000,1000.000000,1000.000000,1000.000000,1000,1000,1000,1000
unique,1000,5,4,6,333,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,5,5,3,13
top,ASSET_0001,Laptop,In Use,Operations,User_005,NaN,NaN,NaN,NaN,Active,NaN,NaN,NaN,NaN,NaN,Dell,Taipei,Compliant,Windows 11
freq,1,319,788,187,6,NaN,NaN,NaN,NaN,606,NaN,NaN,NaN,NaN,NaN,245,300,451,253
mean,NaN,NaN,NaN,NaN,NaN,2023-08-22 09:33:07.199999744,2.662100,2026-10-13 21:38:52.799999744,175.902000,NaN,2025-07-08 03:59:59.999999744,883.24943,2.659000,1.108010,0.250000,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,2020-01-09 00:00:00,0.000000,2022-06-05 00:00:00,-1416.000000,NaN,2021-04-21 00:00:00,20.00000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,2022-05-16 12:00:00,1.300000,2025-07-07 18:00:00,-287.250000,NaN,2024-02-11 00:00:00,155.05250,1.000000,0.420000,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,2023-10-07 00:00:00,2.500000,2026-11-05 00:00:00,198.000000,NaN,2025-08-27 00:00:00,352.66500,2.000000,0.910000,0.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,2024-12-21 00:00:00,3.900000,2028-02-06 00:00:00,656.000000,NaN,2026-11-06 06:00:00,897.06500,4.000000,1.430000,0.250000,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,2026-04-16 00:00:00,6.300000,2031-03-14 00:00:00,1788.000000,NaN,2030-01-14 00:00:00,7600.00000,12.000000,30.000000,1.000000,NaN,NaN,NaN,NaN


In [23]:
df.isnull().sum()

Asset_ID                     0
Asset_Type                   0
Status                       0
Department                   0
Assigned_To                237
Acquisition_Date             0
Asset_Age                    0
Warranty_Expiry_Date         0
Days_to_Warranty_Expiry      0
Warranty_Status              0
License_Expiry_Date        220
Maintenance_Cost             0
Repair_Count                 0
Repair_Intensity             0
Is_High_Cost                 0
Vendor_Name                  0
Location                     0
Compliance_Status            0
OS_Version                   0
dtype: int64

In [24]:
# Save the cleaned dataset to the processed folder
df.to_csv('../data/processed_it_assets.csv', index=False)

## ETL Summary

The dataset has been cleaned, standardized, and enriched through the following pipeline:

- **Deduplication**: Ensured asset uniqueness by removing duplicate `Asset_ID` entries.
- **Handling Missing Values**: Used domain-specific imputation (e.g., asset-type mean for costs) and maintained 'Not Required' tags for peripherals.
- **Logical Alignment**: Standardized OS versions and cleared license metadata for non-computing assets (e.g., Monitors).
- **Advanced Feature Engineering**: Created predictive indicators including `Repair_Intensity` and `Is_High_Cost` to support proactive risk modeling.

### 📊 Data Quality Checklist

| Data Issue | Handling Strategy | Status |
| :--- | :--- | :--- |
| **Asset Redundancy** | Removed duplicate IDs (1 row = 1 unique asset) | ✅ Resolved |
| **Date Formatting** | Unified all date fields to `YYYY-MM-DD` | ✅ Fixed |
| **Logic Errors** | Stripped OS/License info from non-computing assets | ✅ Cleaned |
| **Outliers** | Replaced `Repair_Count` anomalies (value 99) with median | ✅ Resolved |
| **Missing Costs** | Imputed based on `Asset_Type` mean | ✅ Handled |
| **Feature Enrichment** | Calculated **Repair_Intensity** & **Asset_Age** | ✅ Optimized |
| **Target Labeling** | Flagged Top 25% cost assets as **Is_High_Cost** | ✅ Verified |

**The final dataset is consolidated, logically consistent, and ready for predictive analysis.**